In [1]:
import pandas as pd

In [2]:
REQUIRED_LENGTH = 1280

In [3]:
coords = pd.read_table('hg38_fair_CAGE_peaks_phase1and2.bed', header=None, index_col=None, names=['chr', 
                                                                                                       'start', 
                                                                                                       'end', 
                                                                                                       'fid', 
                                                                                                       'score', 
                                                                                                       'strand',
                                                                                                       'start_of_repr',  
                                                                                                       'end_of_repr', 
                                                                                                       'rgb'])

In [4]:
def hg38fantom_transform(row):
    genome, desc = row.fid.split("::")
    if genome != 'hg38':
        return row
    #assert genome == "hg38"
    
    ch, rest = desc.split(":")
    start, rest = rest.split("..")
    start = int(start)
    end, rest = rest.split(",")
    end = int(end)
    strand, rest = rest.split(";")

    dt = row.to_dict()
    dt['chr'] = ch
    dt['start'] = start
    dt['end'] = end
    dt['strand'] = strand
    return pd.Series(dt)

def pad_to_length(start, end, strand, required_length, chrom):
    length = end - start 
    diff = required_length - length
    if diff < 0: # length is greaterr
        if strand == "+":
            start -= diff 
        elif strand == '-':
            end -= diff
        else:
            raise Exception('Wrong strand')
    elif diff > 0:
        lpad, rest = divmod(diff, 2)
        rpad = lpad + rest
        if strand == "+":
            start -= lpad
            end += rpad
        elif strand == '-':
            start -= rpad
            end += lpad
        else:
            raise Exception('Wrong strand')
    if start < 0:
        start = 0
    if end >= len(chrom):
        end = len(chrom)
        #raise NotImplementedError()
    return start, end

def row_pad_to_length(row, required_length, genome):
    start, end = pad_to_length(row.start, row.end, row.strand, required_length, genome[row.chr])
    dt = row.to_dict()
    dt['start'] = start
    dt['end'] = end
    return pd.Series(dt)




In [5]:
coords

,chr,start,end,fid,score,strand,start_of_repr,end_of_repr,rgb
0,chr1,629191,629220,"hg19::chr1:564571..564600,+;hg_1.1",2972,+,629208,629209,"255,0,0"
1,chr1,629259,629269,"hg19::chr1:564639..564649,+;hg_2.1",397,+,629265,629266,"255,0,0"
2,chr1,629886,629898,"hg19::chr1:565266..565278,+;hg_3.1",587,+,629889,629890,"255,0,0"
3,chr1,630098,630103,"hg19::chr1:565478..565483,+;hg_4.1",151,+,630100,630101,"255,0,0"
4,chr1,630129,630161,"hg19::chr1:565509..565541,+;hg_5.1",5172,+,630143,630144,"255,0,0"
...,...,...,...,...,...,...,...,...,...
201290,chrY,21451841,21451849,"hg19::chrY:23613727..23613735,-;hg_201296.1",39,-,21451845,21451846,"0,0,255"
201291,chrY,21456214,21456218,"hg19::chrY:23618100..23618104,-;hg_201297.1",41,-,21456215,21456216,"0,0,255"
201292,chrY,26671129,26671136,"hg19::chrY:28817276..28817283,-;hg_201298.1",49,-,26671132,26671133,"0,0,255"
201293,chrY,56734794,56734819,"hg19::chrY:58856051..58856076,+;hg_201299.1",782,-,56734803,56734804,"255,0,0"


In [6]:
coords = coords.apply(hg38fantom_transform, axis=1)

In [7]:
coords = coords[['chr', 'start', 'end', 'strand']]

In [8]:
coords.head()

,chr,start,end,strand
0,chr1,629191,629220,+
1,chr1,629259,629269,+
2,chr1,629886,629898,+
3,chr1,630098,630103,+
4,chr1,630129,630161,+


In [9]:
from Bio import SeqIO

In [10]:
genome = SeqIO.to_dict(SeqIO.parse("hg38.fa", format='fasta'))


In [11]:
coords = coords.apply(lambda r: row_pad_to_length(r, REQUIRED_LENGTH, genome), axis=1)

In [12]:
coords.head()

,chr,start,end,strand
0,chr1,628566,629846,+
1,chr1,628624,629904,+
2,chr1,629252,630532,+
3,chr1,629461,630741,+
4,chr1,629505,630785,+


In [15]:
coords = coords.rename({"chr": "chrom"}, axis=1)

In [16]:
coords.to_csv("fantom_reg.txt", sep="\t", index=None)

In [17]:
!head fantom_reg.txt

chrom	start	end	strand
chr1	628566	629846	+
chr1	628624	629904	+
chr1	629252	630532	+
chr1	629461	630741	+
chr1	629505	630785	+
chr1	629660	630940	+
chr1	629858	631138	+
chr1	630535	631815	+
chr1	630733	632013	+
